# n4 · 从标量到张量：自动微分  Tensor Autograd

> **能力标签**：SH8（PyTorch 基础 / PyTorch fundamentals）

## 目标 Objectives

完成本课后，你应该能够：

1. 解释**张量**（Tensor）与标量 `Value` 的区别，以及为何需要支持**广播**（broadcasting）。
2. 实现 `_unbroadcast(grad, shape)`，将广播产生的梯度正确归约回原始形状。
3. 推导并实现 **逐元素加法、乘法** 的广播反向传播。
4. 推导并实现 **矩阵乘法**（`@`，matmul）的反向传播：
   - 对 `self`（形状 `(M,K)`）：`out.grad @ B.T`
   - 对 `other`（形状 `(K,N)`）：`A.T @ out.grad`
5. 用 `torch` 做**梯度校验**（gradient check）对照，验证自实现的正确性。

> 对应能力：**SH8**

## 概念讲解 Concepts

### 1. 从标量到张量 From Scalar to Tensor

之前的 `Value` 类只处理**标量**（scalar，0 维数值）。现实中神经网络的每一层都是矩阵运算。
我们把节点的 `.data` 从 `float` 改成 `numpy.ndarray`，同时将 `.grad` 也改成同形状的数组：

| 属性 | 标量 Value | 张量 Tensor |
|------|------|------|
| `data` | `float` | `np.ndarray (任意形状)` |
| `grad` | `float` | `np.ndarray（与 data 同形）` |
| 累加梯度 | `grad += ...` | `grad = grad + ...`（避免 in-place） |

---

### 2. 广播 Broadcasting

NumPy 的广播规则：形状 `(1, 3)` 可与形状 `(4, 3)` 相加，结果形状为 `(4, 3)`。
这带来一个反向传播问题：**输出梯度的形状与输入不同**，必须把梯度"归约"回原始形状。

**广播方向示例：**

```
self.data  shape (1, 3)    →   out.data  shape (4, 3)
           ↑ 广播了 axis=0（4倍），反向时需要对 axis=0 求和
```

---

### 3. `_unbroadcast(grad, shape)`：广播的逆操作

```python
def _unbroadcast(grad, shape):
    # 1. 如果 grad 维数多于 shape，对多余的前置轴求和
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    # 2. shape 中为 1 的维度说明那里被广播过，对那一轴求和并保持维度
    for i, s in enumerate(shape):
        if s == 1 and grad.shape[i] != 1:
            grad = grad.sum(axis=i, keepdims=True)
    return grad
```

---

### 4. 加法反向传播（含广播）

$$z = A + B \implies \frac{\partial L}{\partial A} = \text{unbroadcast}\left(\frac{\partial L}{\partial z}, \text{shape}(A)\right)$$

对 `other` 同理。只有形状变化时才需要归约；若形状相同，`_unbroadcast` 直接返回原梯度。

---

### 5. 乘法反向传播（含广播）

$$z = A \odot B \implies \frac{\partial L}{\partial A} = \text{unbroadcast}\left(B \odot \frac{\partial L}{\partial z}, \text{shape}(A)\right)$$
$$\frac{\partial L}{\partial B} = \text{unbroadcast}\left(A \odot \frac{\partial L}{\partial z}, \text{shape}(B)\right)$$

---

### 6. 矩阵乘法反向传播 Matmul Backward

设 $A$ 形状 $(M, K)$，$B$ 形状 $(K, N)$，$Z = A @ B$ 形状 $(M, N)$，损失 $L$ 对 $Z$ 的梯度为 $\partial Z$（形状 $(M, N)$）。

$$\boxed{\frac{\partial L}{\partial A} = \partial Z @ B^T}  \quad \text{形状: }(M,N) @ (N,K) = (M,K)$$

$$\boxed{\frac{\partial L}{\partial B} = A^T @ \partial Z}  \quad \text{形状: }(K,M) @ (M,N) = (K,N)$$

**记忆口诀**：梯度"夹在"中间，左边拼右边的转置，右边拼左边的转置。

---

### 7. 梯度校验 Gradient Check

用 **数值差分**（numerical gradient）验证 autograd 梯度的正确性：

$$\frac{\partial f}{\partial x_i} \approx \frac{f(x + h \mathbf{e}_i) - f(x - h \mathbf{e}_i)}{2h}, \quad h = 10^{-5}$$

误差应 $< 10^{-5}$。用 PyTorch 做校验时，直接对比 `tensor.grad` 与 `Tensor.grad`。

## 示例 Worked Example

In [ ]:
# ── 完整 Tensor 类（nanograd-l4 镜像）──────────────────────────────────────
import numpy as np


def _unbroadcast(grad, shape):
    """把 grad 归约回 shape（广播逆操作 / inverse of broadcasting)."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for i, s in enumerate(shape):
        if s == 1 and grad.shape[i] != 1:
            grad = grad.sum(axis=i, keepdims=True)
    return grad


class Tensor:
    """NumPy 张量节点，支持自动微分（Tensor node with autograd)."""

    def __init__(self, data, _children=(), _op=""):
        self.data = np.asarray(data, dtype=float)
        self.grad = np.zeros_like(self.data)
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    # ── 加法（含广播）Addition with broadcasting ──────────────────────────
    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad = self.grad + _unbroadcast(out.grad, self.data.shape)
            other.grad = other.grad + _unbroadcast(out.grad, other.data.shape)
        out._backward = _backward
        return out

    # ── 乘法（含广播）Elementwise multiplication ──────────────────────────
    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad = self.grad + _unbroadcast(other.data * out.grad, self.data.shape)
            other.grad = other.grad + _unbroadcast(self.data * out.grad, other.data.shape)
        out._backward = _backward
        return out

    # ── 矩阵乘法 Matmul backward: dA = dZ @ B.T, dB = A.T @ dZ ──────────
    def __matmul__(self, other):
        out = Tensor(self.data @ other.data, (self, other), "@")

        def _backward():
            self.grad = self.grad + out.grad @ other.data.T
            other.grad = other.grad + self.data.T @ out.grad
        out._backward = _backward
        return out

    # ── 全局求和 Sum all elements ─────────────────────────────────────────
    def sum(self):
        out = Tensor(self.data.sum(), (self,), "sum")

        def _backward():
            self.grad = self.grad + np.ones_like(self.data) * out.grad
        out._backward = _backward
        return out

    # ── ReLU ──────────────────────────────────────────────────────────────
    def relu(self):
        out = Tensor(np.maximum(0.0, self.data), (self,), "relu")

        def _backward():
            self.grad = self.grad + (self.data > 0) * out.grad
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1.0

    def backward(self):
        """拓扑排序后反向传播（Topological backpropagation)."""
        topo, visited = [], set()
        def build(v):
            if id(v) not in visited:
                visited.add(id(v))
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = np.ones_like(self.data)
        for v in reversed(topo):
            v._backward()

    def __repr__(self):
        return f"Tensor(shape={self.data.shape}, op='{self._op}')"


print("Tensor 类定义完毕 (Tensor class defined)")


In [ ]:
# ── 广播加法示例：bias 向量加到矩阵每一行 ────────────────────────────────────
# 场景：A 形状 (3, 4)，b 形状 (1, 4)（偏置 bias）
# 正向：out = A + b，形状 (3, 4)
# 反向：db 应为 da_out 在 axis=0 方向求和，回到 (1, 4)

np.random.seed(0)
A = Tensor(np.random.randn(3, 4))
b = Tensor(np.zeros((1, 4)))

out = A + b
loss = out.sum()
loss.backward()

print("广播加法 Broadcasting Addition:")
print(f"  A.shape = {A.data.shape},  b.shape = {b.data.shape}")
print(f"  out.shape = {out.data.shape}")
print(f"  A.grad.shape = {A.grad.shape}  (应为 (3,4))")
print(f"  b.grad.shape = {b.grad.shape}  (应为 (1,4))")
print(f"  b.grad = {b.grad}  (应为全 3.0，即每行都加了 1.0)")

assert A.grad.shape == (3, 4)
assert b.grad.shape == (1, 4)
assert np.allclose(b.grad, np.full((1, 4), 3.0))
print("✓ 广播梯度形状正确 broadcast grad shape OK")


In [ ]:
# ── 矩阵乘法反向传播演示 Matmul Backward ────────────────────────────────────
# A: (2, 3),  B: (3, 4)  →  Z = A @ B: (2, 4)
# dA = dZ @ B.T  →  (2,4)@(4,3) = (2,3)  ✓
# dB = A.T @ dZ  →  (3,2)@(2,4) = (3,4)  ✓

np.random.seed(1)
A = Tensor(np.random.randn(2, 3))
B = Tensor(np.random.randn(3, 4))

Z = A @ B         # (2, 4)
loss = Z.sum()    # 标量损失（dZ = ones(2,4)）
loss.backward()

print("矩阵乘法反向传播 Matmul backward:")
print(f"  A.shape={A.data.shape},  B.shape={B.data.shape},  Z.shape={Z.data.shape}")
print(f"  A.grad.shape = {A.grad.shape}  (期望 (2,3))")
print(f"  B.grad.shape = {B.grad.shape}  (期望 (3,4))")

# 手工计算：dZ = ones(2,4)
dZ = np.ones((2, 4))
expected_dA = dZ @ B.data.T
expected_dB = A.data.T @ dZ

print(f"  max |A.grad - expected| = {np.abs(A.grad - expected_dA).max():.2e}")
print(f"  max |B.grad - expected| = {np.abs(B.grad - expected_dB).max():.2e}")

assert A.grad.shape == (2, 3)
assert B.grad.shape == (3, 4)
assert np.allclose(A.grad, expected_dA, atol=1e-10)
assert np.allclose(B.grad, expected_dB, atol=1e-10)
print("✓ Matmul 反向传播正确 Matmul backward OK")


In [ ]:
# ── PyTorch 梯度校验对照（Oracle Demo）──────────────────────────────────────
# 用 torch 作为权威参考，验证我们的 Tensor 实现与 PyTorch 梯度完全一致
# 若 torch 不可用，此单元格跳过并打印提示（torch is optional oracle）
try:
    import torch
    _HAS_TORCH = True
except ImportError:
    _HAS_TORCH = False
    print("torch 不可用，跳过 PyTorch oracle 对比 (torch not installed, skipping oracle)")

np.random.seed(42)

if _HAS_TORCH:
    def torch_grad_check(A_np, B_np):
        """同时用 nanograd Tensor 和 PyTorch 计算 loss = sum(A @ B + bias)，对比梯度。"""
        bias_np = np.ones((1, B_np.shape[1]))

        # ── nanograd ────────────────────────────────────────────────────────────
        A_t = Tensor(A_np.copy())
        B_t = Tensor(B_np.copy())
        bias_t = Tensor(bias_np.copy())
        loss_t = (A_t @ B_t + bias_t).sum()
        loss_t.backward()

        # ── PyTorch ─────────────────────────────────────────────────────────────
        A_pt = torch.tensor(A_np, dtype=torch.float64, requires_grad=True)
        B_pt = torch.tensor(B_np, dtype=torch.float64, requires_grad=True)
        bias_pt = torch.tensor(bias_np, dtype=torch.float64, requires_grad=True)
        loss_pt = (A_pt @ B_pt + bias_pt).sum()
        loss_pt.backward()

        # ── 对比 ─────────────────────────────────────────────────────────────────
        err_A    = np.abs(A_t.grad    - A_pt.grad.numpy()).max()
        err_B    = np.abs(B_t.grad    - B_pt.grad.numpy()).max()
        err_bias = np.abs(bias_t.grad - bias_pt.grad.numpy()).max()
        return err_A, err_B, err_bias


    A_np = np.random.randn(4, 5)
    B_np = np.random.randn(5, 3)

    err_A, err_B, err_bias = torch_grad_check(A_np, B_np)
    print("PyTorch 梯度校验对照（A(4,5) @ B(5,3) + bias(1,3)):")
    print(f"  max|dA  - torch.dA  | = {err_A:.2e}")
    print(f"  max|dB  - torch.dB  | = {err_B:.2e}")
    print(f"  max|db  - torch.db  | = {err_bias:.2e}")

    assert err_A < 1e-10
    assert err_B < 1e-10
    assert err_bias < 1e-10
    print("✓ 与 PyTorch 梯度完全一致 Matches PyTorch exactly")


## 动手 Exercise

In [ ]:
# ── 动手练习：实现并验证 relu 广播反向传播 ──────────────────────────────────
# 场景：对矩阵 X (形状 (4, 3)) 做 relu，然后求和作为损失
# 任务：运行下面的断言，确认 relu 反向传播对张量也正确

np.random.seed(7)
X = Tensor(np.random.randn(4, 3))

out = X.relu()
loss = out.sum()
loss.backward()

# 手工期望：relu 梯度 = 1 if x>0 else 0
expected_grad = (X.data > 0).astype(float)

print("ReLU 张量反向传播验证:")
print(f"  X.shape = {X.data.shape},  out.shape = {out.data.shape}")
print(f"  X.grad.shape = {X.grad.shape}  (应为 (4,3))")
print(f"  max |X.grad - expected| = {np.abs(X.grad - expected_grad).max():.2e}")

assert X.grad.shape == (4, 3)
assert np.allclose(X.grad, expected_grad)
print("✓ ReLU 张量梯度正确 relu tensor grad OK")

# ── 额外挑战：用 torch 做 relu 梯度校验（torch 可选）───────────────────────
try:
    import torch as _torch
    X_pt = _torch.tensor(X.data, dtype=_torch.float64, requires_grad=True)
    loss_pt = X_pt.relu().sum()
    loss_pt.backward()
    err = np.abs(X.grad - X_pt.grad.numpy()).max()
    print(f"  与 PyTorch 梯度误差 = {err:.2e}")
    assert err < 1e-10
    print("✓ 与 PyTorch 完全一致 Matches PyTorch exactly")
except ImportError:
    print("  torch 不可用，跳过 PyTorch 对比 (torch not installed)")


## 延伸阅读 Further Reading

1. **NumPy Broadcasting Rules**：<https://numpy.org/doc/stable/user/basics.broadcasting.html> — 官方文档，含形状推导规则与可视化示例。
2. **矩阵微积分速查表 Matrix Calculus**：<https://www.matrixcalculus.org/> — 在线工具，输入表达式自动生成矩阵导数。
3. **Goodfellow et al. «Deep Learning»** 附录 D：矩阵微积分基础，含矩阵乘法和 Jacobian 矩阵推导。
4. **PyTorch Autograd Mechanics**：<https://pytorch.org/docs/stable/notes/autograd.html> — PyTorch 如何处理张量梯度，与本课 `Tensor._backward` 直接对应。
5. **Karpathy «micrograd»**：<https://github.com/karpathy/micrograd> — 标量版 autograd 原型；本课是其张量版扩展。

## 关联练习 Related Assignments

```bash
python check.py nanograd-l4
```

> 实现 `Tensor.__add__`、`__mul__`、`__matmul__`、`sum`、`relu` 及 `_unbroadcast`，使广播反向传播与 PyTorch 梯度一致。

> 能力标签：**SH8** · threshold ≥ 0.7